# Data Reprocessing (Pandas)

## Assumptions
- Churn Definition: A user is considered churned if they have a 'Cancellation Confirmation' event in the dataset.
- Time Frame: The analysis will consider user activities in a 7 date window and associate the churn status based on events occurring after this window.


## Cohort Building Logic
- the dataset will identify churned users who have a 'Cancellation Confirmation' event after 2018-10-08 to have at least one week of observation period.
- Non-churned users will match the lookback period of churned users to ensure comparable observation windows using random sampling method (could be further improved).
- Users with insufficient data (less than 7 days of activity, n = 7) will be excluded from the analysis to maintain data integrity.
- The cohort aims to provide a forward predictive view of user behavior leading up to churn events.
- 55 inactive users observed in the random sampled non-churned cohort, there are two options to handle them, each represents a different strategy on apply the model in production:
    1. Keep them in the dataset, as they represent a cross-sectional users segment that may churn, this cross-sectional segment contains users who are inactive, and their inactivity may lead to customer retension in near future. **[Risky]** the dataset may not reflect the real percentage of inactive users in the population, which may bias the model.
    2. Exclude them from the dataset to focus the model on active user behavior, which may provide clearer insights into churn predictors among engaged users and their near future churn behavior. **[Safer]** the dataset will better reflect the real percentage of inactive users in the population, which may reduce the bias of the model.


## Attributes To Be Included in Data Preprocessing and Engineering

* Churn
* Since Registration
* Gender
* Paid Category
* Page event counts
* Device
* Location
* Song stats
* Artist stats
* Session Stats
* Page Stats

### Libraries

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

### Load Data

In [2]:
se_data = pd.read_json("../Data/mini_sparkify_event_data.json",lines=True)
print(se_data.userId.nunique())
se_data = se_data[se_data['userId']!=""]
print(se_data.userId.nunique())
se_data['ts'] = pd.to_datetime(se_data['ts'], unit='ms')
se_data['registration'] = pd.to_datetime(se_data['registration'], unit='ms')

226
225


### Cohort Building

In [3]:
churn_user_date_df = se_data[se_data['page'] == 'Cancellation Confirmation'][['userId', 'ts']].rename(columns={'ts':'cancel_date'})
churn_user_date_df['churn'] = True
churn_user_date_df['include'] = churn_user_date_df['cancel_date'].dt.date >= pd.to_datetime('2018-10-08').date()
churn_user_date_df['start_date'] = None
churn_user_date_df.loc[churn_user_date_df['include'], 'start_date'] = (churn_user_date_df.loc[churn_user_date_df['include'], 'cancel_date'] - pd.Timedelta(days=7)).dt.floor('d')
exclude_churn_users = churn_user_date_df[~churn_user_date_df['include']]['userId'].unique()
churn_user_date_df['end_date'] = churn_user_date_df['cancel_date']
print(churn_user_date_df.shape)

(52, 6)


In [4]:
churn_user_date_df[churn_user_date_df['include']].userId.nunique()

45

In [5]:
se_users = se_data[['userId']].drop_duplicates()
se_users = se_users.merge(churn_user_date_df,how='left')
se_users['include'] = se_users['include'].fillna(True)
se_users['churn'] = se_users['churn'].fillna(False)
start_dates = churn_user_date_df[churn_user_date_df['include']].start_date.tolist()
mask = (se_users['include'])&(se_users.start_date.isna())
n_start_dates_to_assign = se_users[mask].shape[0]
# random sample with replacement from start_dates to assign to users without churn date
se_users.loc[mask, 'start_date'] = pd.Series(start_dates).sample(n=n_start_dates_to_assign, replace=True, random_state=42).values
se_users['start_date'] = pd.to_datetime(se_users['start_date'])
se_users['end_date'] = se_users['start_date'] + pd.Timedelta(days=7)
se_users[se_users['include']].groupby('churn').describe()

/var/folders/0c/_lvyy05j49g_r0hzzfhr03tm0000gn/T/ipykernel_75429/2704280262.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  se_users['include'] = se_users['include'].fillna(True)
/var/folders/0c/_lvyy05j49g_r0hzzfhr03tm0000gn/T/ipykernel_75429/2704280262.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  se_users['churn'] = se_users['churn'].fillna(False)


cancel_date                                                      \
            count                           mean                  min   
churn                                                                   
False           0                            NaT                  NaT   
True           45  2018-10-30 23:33:16.511111168  2018-10-08 21:10:46   

                                                                      \
                       25%                  50%                  75%   
churn                                                                  
False                  NaT                  NaT                  NaT   
True   2018-10-17 07:39:32  2018-10-30 04:59:03  2018-11-12 19:40:08   

                           start_date                                 \
                       max      count                           mean   
churn                                                                  
False                  NaT        173  2018-10-22 20:06:56.184971008   
True   2018-11-29 11:45:09         45            2018-10-23 10:40:00   

                                                                      \
                       min                  25%                  50%   
churn                                                                  
False  2018-10-01 00:00:00  2018-10-10 00:00:00  2018-10-20 00:00:00   
True   2018-10-01 00:00:00  2018-10-10 00:00:00  2018-10-23 00:00:00   

                                                end_date  \
                       75%                  max    count   
churn                                                      
False  2018-11-07 00:00:00  2018-11-22 00:00:00      173   
True   2018-11-05 00:00:00  2018-11-22 00:00:00       45   

                                                           \
                                mean                  min   
churn                                                       
False  2018-10-29 20:06:56.184971008  2018-10-08 00:00:00   
True             2018-10-30 10:40:00  2018-10-08 00:00:00   

                                                                      \
                       25%                  50%                  75%   
churn                                                                  
False  2018-10-17 00:00:00  2018-10-27 00:00:00  2018-11-14 00:00:00   
True   2018-10-17 00:00:00  2018-10-30 00:00:00  2018-11-12 00:00:00   

                            
                       max  
churn                       
False  2018-11-29 00:00:00  
True   2018-11-29 00:00:00

In [6]:
print(f"Total records before filtering: {se_data.shape[0]}")
se_data_sub_activity = se_data[se_data['userId']!=""].merge(se_users[se_users['include']], on='userId', how='inner')
se_data_sub_activity = se_data_sub_activity[(se_data_sub_activity['ts'] >= se_data_sub_activity['start_date']) & (se_data_sub_activity['ts'] <= se_data_sub_activity['end_date'])]
print(f"Total records after filtering: {se_data_sub_activity.shape[0]}")

Total records before filtering: 278154
Total records after filtering: 39624


In [7]:
se_data_sub_activity.userId.nunique()

163

In [8]:
se_data_sub_activity.drop(columns=['cancel_date', 'include', 'start_date', 'end_date']).groupby(['userId', 'churn']).nunique().reset_index().groupby('churn').describe()

ts                                                             \
       count        mean         std  min    25%    50%     75%     max   
churn                                                                     
False  126.0  214.904762  242.894559  1.0   49.0  135.5  252.75  1156.0   
True    37.0  336.081081  321.281668  2.0  143.0  222.0  399.00  1150.0   

      sessionId                                                 page  \
          count      mean       std  min  25%  50%  75%   max  count   
churn                                                                  
False     126.0  2.587302  2.067926  1.0  1.0  2.0  3.0  12.0  126.0   
True       37.0  3.486486  2.155376  1.0  2.0  3.0  5.0   8.0   37.0   

                                                        auth                 \
           mean       std  min  25%   50%   75%   max  count mean  std  min   
churn                                                                         
False  8.547619  3.270736  1.0  6.0   9.0  11.0  16.0  126.0  1.0  0.0  1.0   
True   9.864865  3.392493  1.0  9.0  11.0  12.0  16.0   37.0  1.0  0.0  1.0   

                          method                                               \
       25%  50%  75%  max  count      mean       std  min  25%  50%  75%  max   
churn                                                                           
False  1.0  1.0  1.0  1.0  126.0  1.984127  0.125483  1.0  2.0  2.0  2.0  2.0   
True   1.0  1.0  1.0  1.0   37.0  1.972973  0.164399  1.0  2.0  2.0  2.0  2.0   

      status                                               level            \
       count      mean       std  min  25%  50%  75%  max  count      mean   
churn                                                                        
False  126.0  2.150794  0.420808  1.0  2.0  2.0  2.0  3.0  126.0  1.134921   
True    37.0  2.000000  0.333333  1.0  2.0  2.0  2.0  3.0   37.0  1.189189   

                                         itemInSession              \
            std  min  25%  50%  75%  max         count        mean   
churn                                                                
False  0.343003  1.0  1.0  1.0  1.0  2.0         126.0  145.261905   
True   0.397061  1.0  1.0  1.0  1.0  2.0          37.0  190.081081   

                                                    location                 \
              std  min   25%    50%     75%     max    count mean  std  min   
churn                                                                         
False  157.067434  1.0  37.0   90.0  203.25  1080.0    126.0  1.0  0.0  1.0   
True   178.372485  2.0  69.0  135.0  297.00   868.0     37.0  1.0  0.0  1.0   

                          userAgent                                     \
       25%  50%  75%  max     count mean  std  min  25%  50%  75%  max   
churn                                                                    
False  1.0  1.0  1.0  1.0     126.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   
True   1.0  1.0  1.0  1.0      37.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   

      lastName                                    firstName                 \
         count mean  std  min  25%  50%  75%  max     count mean  std  min   
churn                                                                        
False    126.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0     126.0  1.0  0.0  1.0   
True      37.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0      37.0  1.0  0.0  1.0   

                          registration                                     \
       25%  50%  75%  max        count mean  std  min  25%  50%  75%  max   
churn                                                                       
False  1.0  1.0  1.0  1.0        126.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   
True   1.0  1.0  1.0  1.0         37.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   

      gender                                    artist              \
       count mean  std  min  25%  50%  75%  max  count        mean   
churn                                        

In [9]:
se_users['has_activity'] = se_users['userId'].isin(se_data_sub_activity['userId'])
se_users['has_activity'].value_counts()

has_activity
True     163
False     62
Name: count, dtype: int64

### Data Preprocessing & Feature Engineering

**Label**
* Churn ✅

**persona**
* Gender ✅
* Location ✅

**Account**
* Since Registration ✅
* Paid Category ✅

**Activities / Behavioral**
* Device
* Page event counts
* Session event counts

**Preferences**
* Song stats
* Artist stats

#### Label & Cohort Main DataFrame

In [10]:
se_data_in_scope_user = se_data[se_data['userId'].isin(se_users[se_users['include']]['userId'])].copy()


In [11]:
se_cohort_df = se_users[se_users['include']].drop(columns=['include']).copy()
se_cohort_df.shape

(218, 6)

#### Persona

In [12]:
unique_ids = se_data['userId'].unique()
excl_ids = se_users[~se_users['include']]['userId'].unique().tolist()
print(f"total userIds: {len(unique_ids)}")
print(f"empty userIds: {len([uid for uid in unique_ids if uid == ''])}")
print(f"excluded churned userIds: {len(excl_ids)}")
print(f"final cohort userIds: {len(unique_ids) - len([uid for uid in unique_ids if uid == '']) - len(excl_ids)}")
# total userId: 226; 
# minus 1 empty userId '', 225 remaining;
# exclude 7 churned users who cancelled before 2018-10-08, final cohort userIds: 218

total userIds: 225
empty userIds: 0
excluded churned userIds: 7
final cohort userIds: 218


In [13]:
persona_cols = ['userId','gender', 'location', 'registration']
se_data[(se_data['userId'] != '') & (se_data['userId'].isin(se_users[se_users['include']]['userId']))][persona_cols]\
    .drop_duplicates()\
    .groupby('userId').nunique().reset_index().describe()
# all users have unique profile info

,gender,location,registration
count,218.0,218.0,218.0
mean,1.0,1.0,1.0
std,0.0,0.0,0.0
min,1.0,1.0,1.0
25%,1.0,1.0,1.0
50%,1.0,1.0,1.0
75%,1.0,1.0,1.0
max,1.0,1.0,1.0


In [14]:
se_persona = se_users[se_users['include']][['userId','end_date']].merge(
    se_data[(se_data['userId'] != '') & (se_data['userId'].isin(se_users[se_users['include']]['userId']))][persona_cols]\
    .drop_duplicates(), how='inner', on='userId'
)
se_persona['day_since_reg'] = (se_persona['end_date'] - se_persona['registration']).dt.days
print(f"persona df shape: {se_persona.shape}")
se_persona[['city','state']] = se_persona['location'].str.split(',', expand=True)
se_persona['city'] = se_persona['city'].str.strip().str.title()
se_persona['state'] = se_persona['state'].str.strip().str.upper()
se_persona = se_persona[['userId','gender','city','state','day_since_reg']]\
    .rename(columns={'gender':'psn_gender','city':'psn_city','state':'psn_state','day_since_reg':'acct_day_since_reg'})

persona df shape: (218, 6)


In [15]:
# Feature engineering
se_persona['psn_multi_city'] = se_persona['psn_city'].str.contains('-')
se_persona['psn_multi_state'] = se_persona['psn_state'].str.contains('-')

In [16]:
se_persona
# location definitions require further cleaning:
# - some cities have multiple names separated by '-', some separated by '--'
# - some states have multiple abbreviations separated by '-'
# - by default multi_state should have multi_city records, but there are some exceptions

,userId,psn_gender,psn_city,psn_state,acct_day_since_reg,psn_multi_city,psn_multi_state
0,30,M,Bakersfield,CA,34,False,False
1,9,M,Boston-Cambridge-Newton,MA-NH,31,True,True
2,74,F,Tallahassee,FL,40,False,False
3,54,F,Spokane-Spokane Valley,WA,109,True,False
4,4,M,Baltimore-Columbia-Towson,MD,48,True,False
5,101,M,Denver-Aurora-Lakewood,CO,53,True,False
6,78,F,Mcallen-Edinburg-Mission,TX,16,True,False
7,88,F,Columbus,GA-AL,63,False,True
8,95,F,Phoenix-Mesa-Scottsdale,AZ,33,True,False
9,25,F,Tampa-St. Petersburg-Clearwater,FL,65,True,False


#### Account

In [17]:
se_data_in_scope_user.groupby('userId')['level'].nunique().reset_index().describe()

,level
count,218.000000
mean,1.619266
std,0.486685
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,2.000000


In [18]:
act_before_end_df = se_data_in_scope_user.merge(se_users,how='left',on='userId').query('end_date > ts')
se_last_level = (
    act_before_end_df[~act_before_end_df['page'].str.lower().str.contains('cancel')]
    .sort_values(['userId', 'ts'])
    .groupby('userId')['level']
    .agg(last_level='last', multiple_levels=lambda x: x.nunique() > 1)
    .reset_index()
)
print(se_last_level.shape)
# there are only 202 user IDs in the filtered data period, the rest 16 user does not have level info before end_date, interpolate nearest datapoint from the dataset, and multiple_level as false as there was no level change

se_last_level

(202, 3)


,userId,last_level,multiple_levels
0,10,paid,False
1,100,free,True
2,100002,paid,False
3,100003,free,False
4,100004,free,False
5,100005,free,False
6,100007,paid,False
7,100008,paid,False
8,100009,free,True
9,100010,free,False


In [19]:
rest_level_df = se_data_in_scope_user[~se_data_in_scope_user['userId'].isin(se_last_level.userId)][['userId','ts','level']] \
    .merge(se_users[['userId','end_date']],how='left',on='userId').copy()
rest_level_df['time_diff'] = (rest_level_df['end_date'] - rest_level_df['ts']).abs()
rest_level_nearest = rest_level_df.sort_values(['userId','time_diff']).groupby('userId').first().reset_index()[['userId','level']].rename(columns={'level':'last_level'})
rest_level_nearest['multiple_levels'] = False
rest_level_nearest

,userId,last_level,multiple_levels
0,100017,free,False
1,116,free,False
2,123,free,False
3,125,free,False
4,14,paid,False
5,151,paid,False
6,152,free,False
7,155,free,False
8,156,free,False
9,2,paid,False


In [20]:
se_last_level = pd.concat([se_last_level, rest_level_nearest], ignore_index=True)
se_last_level = se_last_level.rename(columns={'last_level':'acct_last_level','multiple_levels':'acct_multiple_levels'})
se_last_level.shape

(218, 3)

#### Activities / Behavioral

In [21]:
se_page = se_data_sub_activity.loc[~se_data_sub_activity['page'].str.lower().str.contains('cancel'), ['userId','page']]\
    .assign(page=lambda x: 'act_page_' + x['page'].str.replace(' ', '_').str.lower())\
    .pivot_table(index='userId', columns='page', aggfunc='size', fill_value=0)\
    .reset_index()
se_page_agg = se_data_sub_activity.loc[~se_data_sub_activity['page'].str.lower().str.contains('cancel'), ['userId','page']]\
    .groupby('userId')['page'].agg(['count','nunique']).reset_index().rename(columns={'count':'total_page_visits', 'nunique':'unique_page_types'})
se_page = se_page.merge(se_page_agg, how='left', on='userId')
print(se_page.shape)
se_page = se_cohort_df[['userId']].merge(se_page, how='left', on='userId').fillna(0)
print(se_page.shape)
for col in se_page.columns[1:]:
    se_page[col] = se_page[col].astype(int)
se_page

(163, 20)
(218, 20)


,userId,act_page_about,act_page_add_friend,act_page_add_to_playlist,act_page_downgrade,act_page_error,act_page_help,act_page_home,act_page_logout,act_page_nextsong,act_page_roll_advert,act_page_save_settings,act_page_settings,act_page_submit_downgrade,act_page_submit_upgrade,act_page_thumbs_down,act_page_thumbs_up,act_page_upgrade,total_page_visits,unique_page_types
0,30,1,3,5,0,0,1,9,4,221,18,0,4,0,0,2,9,4,281,12
1,9,0,4,3,2,1,2,6,1,158,6,0,0,1,1,1,10,2,198,14
2,74,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1
3,54,2,5,11,13,0,4,21,5,526,0,1,5,0,0,8,36,0,637,12
4,4,1,14,12,4,2,6,17,3,437,0,0,2,0,0,4,24,0,526,12
5,101,1,9,37,15,0,3,47,15,917,1,2,7,0,0,11,48,0,1113,13
6,78,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,88,1,2,8,4,0,2,7,1,230,0,1,2,0,0,3,12,0,273,12
8,95,0,0,0,0,1,0,1,2,64,4,0,0,0,0,1,3,1,77,8
9,25,0,6,15,4,0,1,26,6,495,5,0,2,1,0,4,25,0,590,12


In [22]:
se_session = se_data_sub_activity[['userId','sessionId']]\
    .groupby('userId')['sessionId']\
    .agg(['nunique','mean'])\
    .reset_index()\
    .rename(columns={'nunique':'session_count', 'mean':'session_avg_pages'})
print(se_session.shape)
se_session = se_session.merge(se_cohort_df[['userId']], how='right', on='userId').fillna(0)
print(se_session.shape)
se_session

(163, 3)
(218, 3)


,userId,session_count,session_avg_pages
0,30,5.0,1353.619217
1,9,3.0,1323.772727
2,74,1.0,1173.000000
3,54,7.0,1693.499215
4,4,2.0,1746.277567
5,101,4.0,658.261456
6,78,0.0,0.000000
7,88,2.0,1682.666667
8,95,2.0,1324.714286
9,25,3.0,1542.688136


In [23]:
se_days_active = se_data_sub_activity[['userId','ts']]\
    .assign(activity_day=lambda x: x['ts'].dt.floor('d'))\
    .groupby('userId')['activity_day'].nunique().reset_index().rename(columns={'activity_day':'act_days_active'})
print(se_days_active.shape)
se_days_active = se_days_active.merge(se_cohort_df[['userId']], how='right', on='userId').fillna(0)
print(se_days_active.shape)
se_days_active

(163, 2)
(218, 2)


,userId,act_days_active
0,30,4.0
1,9,4.0
2,74,1.0
3,54,6.0
4,4,2.0
5,101,6.0
6,78,0.0
7,88,2.0
8,95,1.0
9,25,3.0


#### Preferences

In [24]:
# all song artist data preparation
se_data_song_artist = se_data[['artist','song']].dropna().copy()
se_data_song_artist['song'] = se_data_song_artist['song'].str.strip().str.lower()
se_data_song_artist['artist'] = se_data_song_artist['artist'].str.strip().str.lower()
se_data_song_artist['song_artist'] = se_data_song_artist['artist'] + " - " + se_data_song_artist['song']

In [25]:
song_artist_counts = se_data_song_artist['song_artist'].value_counts().reset_index().sort_values('count', ascending=False).reset_index(drop=True)
song_artist_counts['prf_top100song'] = song_artist_counts['song_artist'].isin(song_artist_counts['song_artist'][:100])
song_artist_counts['prf_top1000song'] = song_artist_counts['song_artist'].isin(song_artist_counts['song_artist'][:1000])
song_artist_counts['prf_unique_song'] = song_artist_counts['count'] == 1
print(song_artist_counts.shape)
song_artist_counts[['prf_top100song','prf_top1000song','prf_unique_song']].sum()

(65402, 5)


prf_top100song       100
prf_top1000song     1000
prf_unique_song    36095
dtype: int64

In [26]:
artist_counts = se_data_song_artist['artist'].value_counts().reset_index().sort_values('count', ascending=False).reset_index(drop=True)
artist_counts['prf_top20artist'] = artist_counts['artist'].isin(artist_counts['artist'][:20])
artist_counts['prf_top100artist'] = artist_counts['artist'].isin(artist_counts['artist'][:100])
artist_counts['prf_unique_artist'] = artist_counts['count'] == 1
print(artist_counts.shape)
artist_counts[['prf_top20artist','prf_top100artist','prf_unique_artist']].sum()

(17655, 5)


prf_top20artist        20
prf_top100artist      100
prf_unique_artist    5809
dtype: int64

In [27]:
song_artist_meta_data = se_data_song_artist.drop_duplicates()\
    .merge(artist_counts.drop(columns=['count'])).merge(song_artist_counts.drop(columns=['count']))
song_artist_meta_data

,artist,song,song_artist,prf_top20artist,prf_top100artist,prf_unique_artist,prf_top100song,prf_top1000song,prf_unique_song
0,martha tilston,rockpools,martha tilston - rockpools,False,False,False,False,False,False
1,five iron frenzy,canada,five iron frenzy - canada,False,True,False,True,True,False
2,adam lambert,time for miracles,adam lambert - time for miracles,False,False,False,False,False,False
3,enigma,knocking on forbidden doors,enigma - knocking on forbidden doors,False,False,False,False,True,False
4,daft punk,harder better faster stronger,daft punk - harder better faster stronger,True,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...
65397,tommy james and the shondells,she,tommy james and the shondells - she,False,False,False,False,False,True
65398,junior kelly,black woman,junior kelly - black woman,False,False,False,False,False,True
65399,shaggy,lucky day,shaggy - lucky day,False,False,False,False,False,True
65400,saliva,king of the stereo,saliva - king of the stereo,False,False,False,False,False,True


In [28]:
se_prf_raw = se_data_sub_activity[['userId','song','artist']].dropna()\
    .assign(
        song=lambda x: x['song'].str.strip().str.lower(),
        artist=lambda x: x['artist'].str.strip().str.lower()
    )\
    .merge(song_artist_meta_data, how='left', on=['artist','song'])
se_prf_counts = se_prf_raw[['userId','song_artist']].groupby('userId').nunique().reset_index().rename(columns={'song_artist':'prf_unique_songs_count'})
se_prf_top = se_prf_raw[['userId','prf_top20artist','prf_top100artist','prf_unique_artist','prf_top100song','prf_top1000song','prf_unique_song']]\
    .groupby('userId').agg(['sum','any']).reset_index()
se_prf_top.columns = [
    '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
    for col in se_prf_top.columns
]
print(se_prf_top.shape)
se_prf_top = se_prf_top.merge(se_cohort_df[['userId']], how='right', on='userId')
se_prf_top.columns = [col.replace('_sum','_count') for col in se_prf_top.columns]
any_cols = [col for col in se_prf_top.columns if col.endswith('_any')]
se_prf_top[any_cols] = se_prf_top[any_cols].fillna(False)
cnt_cols = [col for col in se_prf_top.columns if col.endswith('_count')]
se_prf_top[cnt_cols] = se_prf_top[cnt_cols].fillna(0).astype(int)
print(se_prf_top.shape)
se_prf_top

(162, 13)
(218, 13)


/var/folders/0c/_lvyy05j49g_r0hzzfhr03tm0000gn/T/ipykernel_75429/1944254836.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  se_prf_top[any_cols] = se_prf_top[any_cols].fillna(False)


,userId,prf_top20artist_count,prf_top20artist_any,prf_top100artist_count,prf_top100artist_any,prf_unique_artist_count,prf_unique_artist_any,prf_top100song_count,prf_top100song_any,prf_top1000song_count,prf_top1000song_any,prf_unique_song_count,prf_unique_song_any
0,30,23,True,56,True,12,True,18,True,60,True,35,True
1,9,17,True,43,True,0,False,19,True,45,True,14,True
2,74,0,False,0,False,0,False,0,False,0,False,0,False
3,54,50,True,122,True,20,True,52,True,151,True,99,True
4,4,46,True,100,True,11,True,50,True,130,True,73,True
5,101,73,True,183,True,21,True,84,True,240,True,149,True
6,78,0,False,0,False,0,False,0,False,0,False,0,False
7,88,23,True,56,True,3,True,24,True,57,True,26,True
8,95,2,True,12,True,1,True,5,True,13,True,14,True
9,25,51,True,116,True,10,True,51,True,138,True,77,True


#### Merge All Features

In [29]:
print(f"Persona shape: {se_persona.shape}")
print(f"Last level shape: {se_last_level.shape}")
print(f"Page shape: {se_page.shape}")
print(f"Session shape: {se_session.shape}")
print(f"Days active shape: {se_days_active.shape}")
print(f"Preference shape: {se_prf_top.shape}")

in_scope_user_ids = se_cohort_df['userId'].unique()
active_user_ids = se_data_sub_activity['userId'].unique()
print(f"Total in-scope users: {len(in_scope_user_ids)}")
print(f"Total active users in filtered period: {len(active_user_ids)}")
print(f"""Persona, all in-scope users found: {sum(se_persona['userId'].isin(in_scope_user_ids)) == len(in_scope_user_ids)}, 
      all active users found: {sum(se_persona['userId'].isin(active_user_ids)) == len(active_user_ids)} """)
print(f"""Last level, all in-scope users found: {sum(se_last_level['userId'].isin(in_scope_user_ids)) == len(in_scope_user_ids)}, 
      all active users found: {sum(se_last_level['userId'].isin(active_user_ids)) == len(active_user_ids)} """)
print(f"""Page, all in-scope users found: {sum(se_page['userId'].isin(in_scope_user_ids)) == len(in_scope_user_ids)}, 
      all active users found: {sum(se_page['userId'].isin(active_user_ids)) == len(active_user_ids)} """)
print(f"""Session, all in-scope users found: {sum(se_session['userId'].isin(in_scope_user_ids)) == len(in_scope_user_ids)}, 
      all active users found: {sum(se_session['userId'].isin(active_user_ids)) == len(active_user_ids)} """)
print(f"""Days active, all in-scope users found: {sum(se_days_active['userId'].isin(in_scope_user_ids)) == len(in_scope_user_ids)}, 
      all active users found: {sum(se_days_active['userId'].isin(active_user_ids)) == len(active_user_ids)} """)
print(f"""Preference, all in-scope users found: {sum(se_prf_top['userId'].isin(in_scope_user_ids)) == len(in_scope_user_ids)}, 
      all active users found: {sum(se_prf_top['userId'].isin(active_user_ids)) == len(active_user_ids)} """)

Persona shape: (218, 7)
Last level shape: (218, 3)
Page shape: (218, 20)
Session shape: (218, 3)
Days active shape: (218, 2)
Preference shape: (218, 13)
Total in-scope users: 218
Total active users in filtered period: 163
Persona, all in-scope users found: True, 
      all active users found: True 
Last level, all in-scope users found: True, 
      all active users found: True 
Page, all in-scope users found: True, 
      all active users found: True 
Session, all in-scope users found: True, 
      all active users found: True 
Days active, all in-scope users found: True, 
      all active users found: True 
Preference, all in-scope users found: True, 
      all active users found: True 


In [30]:
print(f"Cohort df shape: {se_cohort_df.columns.tolist()}")
print(f"Persona shape: {se_persona.columns.tolist()}")
print(f"Last level shape: {se_last_level.columns.tolist()}")
print(f"Page shape: {se_page.columns.tolist()}")
print(f"Session shape: {se_session.columns.tolist()}")
print(f"Days active shape: {se_days_active.columns.tolist()}")
print(f"Preference shape: {se_prf_top.columns.tolist()}")

Cohort df shape: ['userId', 'cancel_date', 'churn', 'start_date', 'end_date', 'has_activity']
Persona shape: ['userId', 'psn_gender', 'psn_city', 'psn_state', 'acct_day_since_reg', 'psn_multi_city', 'psn_multi_state']
Last level shape: ['userId', 'acct_last_level', 'acct_multiple_levels']
Page shape: ['userId', 'act_page_about', 'act_page_add_friend', 'act_page_add_to_playlist', 'act_page_downgrade', 'act_page_error', 'act_page_help', 'act_page_home', 'act_page_logout', 'act_page_nextsong', 'act_page_roll_advert', 'act_page_save_settings', 'act_page_settings', 'act_page_submit_downgrade', 'act_page_submit_upgrade', 'act_page_thumbs_down', 'act_page_thumbs_up', 'act_page_upgrade', 'total_page_visits', 'unique_page_types']
Session shape: ['userId', 'session_count', 'session_avg_pages']
Days active shape: ['userId', 'act_days_active']
Preference shape: ['userId', 'prf_top20artist_count', 'prf_top20artist_any', 'prf_top100artist_count', 'prf_top100artist_any', 'prf_unique_artist_count', 'p

In [31]:
se_data_processed = se_cohort_df\
    .merge(se_persona, how='left', on='userId')\
    .merge(se_last_level, how='left', on='userId')\
    .merge(se_page, how='left', on='userId')\
    .merge(se_session, how='left', on='userId')\
    .merge(se_days_active, how='left', on='userId')\
    .merge(se_prf_top, how='left', on='userId')

print(f"Final processed data shape: {se_data_processed.shape}")
print(f"Users with Activities: {(se_data_processed['has_activity']).sum()}, inactive users: {se_data_processed.shape[0] - (se_data_processed['has_activity']).sum()}")
print(f"Churned users: {se_data_processed['churn'].sum()}, non-churned users: {se_data_processed.shape[0] - se_data_processed['churn'].sum()}")
se_data_processed.to_csv("../Output/data_proc_eng/se_data_processed_pd.csv", index=False,encoding='utf-8')

Final processed data shape: (218, 48)
Users with Activities: 163, inactive users: 55
Churned users: 45, non-churned users: 173
